# Lab 21 — QLoRA Fine-tuning Router Agent cho **Chatbot Tuyển sinh** · T4 / Colab

Notebook này là bản đã chỉnh cho product chatbot tuyển sinh của bạn.

**Mục tiêu**: fine-tune một **router agent** để nhận câu hỏi user và trả về JSON route:

```json
{
  "intent": "tuition_lookup | scholarship_lookup | timeline_process | admission_requirement | program_info | school_info | general_question",
  "answer_mode": "direct | clarify | retrieve",
  "needs_retrieval": true,
  "needs_clarification": false,
  "clarification_question": null,
  "transformed_query": "..."
}
```

**Ý nghĩa với product của bạn**:

```text
Prompt-based router hiện tại → dùng để gán nhãn nháp / collect dataset
Fine-tuned router sau lab → thay bước gọi LLM lớn để đoán intent/answer_mode
RAG/Qdrant vẫn giữ nhiệm vụ lấy kiến thức tuyển sinh chính xác
```

## Profile T4

| Setting | Giá trị |
|---|---|
| Base model | `unsloth/Qwen2.5-3B-bnb-4bit` |
| Fine-tune | QLoRA 4-bit |
| Library | Unsloth + TRL SFTTrainer + PEFT |
| Rank experiment | `r=8`, `r=16`, `r=64` |
| Dataset | Synthetic admissions-router dataset, 200–500 samples |
| Output | LoRA adapters + CSV metrics + qualitative comparison |

> Chạy trên Colab: `Runtime > Change runtime type > GPU`.


## 0. Setup & Environment Check


In [ ]:
# Verify GPU is available before installing anything
!nvidia-smi


In [ ]:
import torch
assert torch.cuda.is_available(), "❌ GPU runtime chưa bật. Vào Runtime > Change runtime type > GPU"
name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU: {name}")
print(f"✓ VRAM: {vram_gb:.1f} GB")
print(f"✓ CUDA: {torch.version.cuda}")
print(f"✓ PyTorch: {torch.__version__}")

if vram_gb > 20:
    print("\n⚠️ GPU lớn hơn T4. Bạn vẫn chạy được notebook này, nhưng có thể tăng batch hoặc dùng model 7B.")
elif "T4" not in name:
    print(f"\n💡 Notebook này tối ưu cho T4 16GB. GPU hiện tại: {name}")


In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.12,<0.16" peft accelerate bitsandbytes
!pip install -q datasets matplotlib pandas numpy huggingface_hub


In [ ]:
# Optional: mount Google Drive để lưu checkpoint không bị mất khi Colab disconnect
MOUNT_DRIVE = False  # đổi thành True nếu muốn save vào Drive

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/lab21_admissions_router_qlora'
else:
    OUTPUT_DIR = '/content/lab21_admissions_router_qlora'

import os, json, random, time, gc, re
from pathlib import Path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output dir: {OUTPUT_DIR}")


## 1. Dataset cho Chatbot Tuyển sinh

Notebook này tạo dataset **router agent** dạng Alpaca:

```json
{
  "instruction": "Bạn là router agent...",
  "input": "Ngành AI học phí bao nhiêu?",
  "output": "{\"intent\":\"tuition_lookup\", ... }"
}
```

Dataset này **không dạy model trả lời học phí/deadline cụ thể**. Nó chỉ dạy model chọn hướng xử lý:

```text
direct   → chào hỏi/cảm ơn/câu đơn giản
clarify  → câu hỏi thiếu ngành/bậc/năm
retrieve → cần gọi RAG/Qdrant để lấy tài liệu tuyển sinh
```

Sau này khi có log thật từ DB, bạn thay cell tạo synthetic dataset bằng file export `router_dataset_v1.jsonl` từ admin review.


In [ ]:
from datasets import Dataset

ROUTER_INSTRUCTION = """Bạn là router agent cho chatbot tuyển sinh VinUniversity.
Nhiệm vụ: phân loại câu hỏi của user và quyết định hướng xử lý tiếp theo.
Chỉ trả về JSON hợp lệ, không thêm markdown, không giải thích.

Allowed intent:
- tuition_lookup
- scholarship_lookup
- timeline_process
- admission_requirement
- program_info
- school_info
- general_question

Allowed answer_mode:
- direct: dùng cho chào hỏi, cảm ơn, xác nhận ngắn, hoặc câu không cần tra cứu dữ liệu tuyển sinh
- clarify: dùng khi thiếu thông tin quan trọng như ngành, bậc học, năm tuyển sinh
- retrieve: dùng khi cần tra cứu dữ liệu tuyển sinh từ RAG/Qdrant

JSON schema bắt buộc:
{
  "intent": string,
  "answer_mode": string,
  "needs_retrieval": boolean,
  "needs_clarification": boolean,
  "clarification_question": string | null,
  "transformed_query": string | null
}
""".strip()

INTENTS = [
    "tuition_lookup",
    "scholarship_lookup",
    "timeline_process",
    "admission_requirement",
    "program_info",
    "school_info",
    "general_question",
]
ANSWER_MODES = ["direct", "clarify", "retrieve"]

MAJORS = [
    "Khoa học Máy tính", "Trí tuệ Nhân tạo", "Data Science", "Kỹ thuật Điện",
    "Kỹ thuật Cơ khí", "Quản trị Kinh doanh", "Marketing", "Tài chính",
    "Điều dưỡng", "Y khoa", "Bác sĩ Nội trú", "Thạc sĩ Khoa học Máy tính",
]
LEVELS = ["Cử nhân", "Thạc sĩ", "Tiến sĩ", "Sau đại học"]
YEARS = ["2025", "2026", "năm nay", "năm sau"]

random.seed(42)

def dumps(obj):
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))

def sample(question, intent, answer_mode, needs_retrieval, needs_clarification, clarification_question=None, transformed_query=None):
    return {
        "instruction": ROUTER_INSTRUCTION,
        "input": question,
        "output": dumps({
            "intent": intent,
            "answer_mode": answer_mode,
            "needs_retrieval": bool(needs_retrieval),
            "needs_clarification": bool(needs_clarification),
            "clarification_question": clarification_question,
            "transformed_query": transformed_query,
        })
    }

samples = []

# 1) tuition_lookup — retrieve khi đủ/khá đủ thông tin
for major in MAJORS:
    for q in [
        f"Ngành {major} học phí bao nhiêu?",
        f"Cho em hỏi học phí {major} ở VinUni là bao nhiêu?",
        f"Chi phí học ngành {major} năm 2026 thế nào?",
        f"Em muốn biết tuition của chương trình {major}.",
    ]:
        samples.append(sample(
            q, "tuition_lookup", "retrieve", True, False,
            None, f"học phí {major} VinUni"
        ))

# 2) tuition_lookup — clarify khi thiếu slot
for q in [
    "Học phí bao nhiêu?",
    "Cho em hỏi tiền học ở VinUni?",
    "Một năm học hết bao nhiêu tiền ạ?",
    "Chi phí học tập thế nào?",
    "Tuition fee là bao nhiêu vậy?",
    "Học phí chương trình này như thế nào?",
]:
    samples.append(sample(
        q, "tuition_lookup", "clarify", False, True,
        "Bạn đang muốn hỏi học phí của ngành nào và bậc học nào tại VinUni?",
        None
    ))

# 3) scholarship_lookup
for major in MAJORS[:10]:
    for q in [
        f"Ngành {major} có học bổng không?",
        f"Em muốn hỏi học bổng cho {major}.",
        f"Scholarship của ngành {major} cần điều kiện gì?",
    ]:
        samples.append(sample(
            q, "scholarship_lookup", "retrieve", True, False,
            None, f"học bổng {major} VinUni"
        ))

for q in [
    "VinUni có học bổng không?",
    "Em muốn xin hỗ trợ tài chính thì làm sao?",
    "Học bổng cần IELTS bao nhiêu?",
    "Có khoản vay sinh viên không?",
    "Financial aid của VinUni như thế nào?",
]:
    samples.append(sample(
        q, "scholarship_lookup", "retrieve", True, False,
        None, "học bổng hỗ trợ tài chính VinUni"
    ))

# 4) timeline_process
for q in [
    "Deadline nộp hồ sơ VinUni là khi nào?",
    "Khi nào hết hạn đăng ký tuyển sinh?",
    "Quy trình apply vào VinUni gồm mấy bước?",
    "Em muốn biết timeline tuyển sinh năm 2026.",
    "Bao giờ có kết quả tuyển sinh?",
    "Lịch phỏng vấn tuyển sinh là khi nào?",
    "Nộp hồ sơ online ở đâu và các bước ra sao?",
    "Admission timeline của VinUni năm nay thế nào?",
]:
    samples.append(sample(
        q, "timeline_process", "retrieve", True, False,
        None, "deadline quy trình timeline tuyển sinh VinUni"
    ))

# 5) admission_requirement
for major in MAJORS[:10]:
    for q in [
        f"Điều kiện tuyển sinh ngành {major} là gì?",
        f"Ngành {major} cần IELTS/SAT không?",
        f"Yêu cầu đầu vào của {major} ở VinUni là gì?",
    ]:
        samples.append(sample(
            q, "admission_requirement", "retrieve", True, False,
            None, f"điều kiện tuyển sinh yêu cầu đầu vào {major} VinUni"
        ))

for q in [
    "Điều kiện tuyển sinh là gì?",
    "Cần những giấy tờ gì?",
    "Yêu cầu đầu vào thế nào?",
    "Có cần IELTS không?",
]:
    samples.append(sample(
        q, "admission_requirement", "clarify", False, True,
        "Bạn đang muốn hỏi điều kiện tuyển sinh cho ngành nào và bậc học nào tại VinUni?",
        None
    ))

# 6) program_info
for major in MAJORS:
    for q in [
        f"Ngành {major} học những gì?",
        f"Chương trình {major} kéo dài bao lâu?",
        f"{major} thuộc viện nào của VinUni?",
        f"Cơ hội nghề nghiệp ngành {major} thế nào?",
    ]:
        samples.append(sample(
            q, "program_info", "retrieve", True, False,
            None, f"thông tin chương trình ngành {major} VinUni"
        ))

for q in [
    "VinUni có những ngành nào?",
    "Có ngành công nghệ thông tin không?",
    "Các chương trình cử nhân gồm những gì?",
    "Sau đại học có chương trình nào?",
]:
    samples.append(sample(
        q, "program_info", "retrieve", True, False,
        None, "danh sách ngành chương trình đào tạo VinUni"
    ))

# 7) school_info
for q in [
    "VinUni là trường gì?",
    "VinUniversity nằm ở đâu?",
    "Trường có ký túc xá không?",
    "VinUni đào tạo bằng tiếng Anh hay tiếng Việt?",
    "Thông tin chung về VinUni.",
    "VinUni có liên kết quốc tế không?",
]:
    samples.append(sample(
        q, "school_info", "retrieve", True, False,
        None, "thông tin chung VinUniversity VinUni"
    ))

# 8) general/direct
for q in [
    "Xin chào", "Chào bạn", "Hello", "Hi bot", "Bạn có thể giúp gì cho mình?",
    "Cảm ơn bạn nhé", "Ok cảm ơn", "Tks", "Thank you", "Được rồi", "Ừ", "Vâng",
    "Mình tên là Hải", "Tôi đang học lớp 12", "Mình quan tâm VinUni", "Bạn là ai?",
]:
    samples.append(sample(
        q, "general_question", "direct", False, False,
        None, None
    ))

# 9) mixed / noisy Vietnamese typing
noisy_cases = [
    ("hoc phi nganh ai bao nhieu", "tuition_lookup", "retrieve", "học phí ngành AI VinUni"),
    ("hoc bong vinuni co kho khong", "scholarship_lookup", "retrieve", "học bổng VinUni điều kiện"),
    ("deadline nop ho so la khi nao", "timeline_process", "retrieve", "deadline nộp hồ sơ VinUni"),
    ("nganh data science can gi", "admission_requirement", "retrieve", "điều kiện tuyển sinh ngành Data Science VinUni"),
    ("vinuni co nganh nao", "program_info", "retrieve", "danh sách ngành VinUni"),
]
for q, intent, mode, tq in noisy_cases:
    samples.append(sample(q, intent, mode, True, False, None, tq))

# Shuffle + cap to 260 examples for T4 speed
random.shuffle(samples)
MAX_SAMPLES = 260
samples = samples[:MAX_SAMPLES]

raw = Dataset.from_list(samples)
print(f"✓ Dataset size: {len(raw)}")
print(f"✓ Columns: {raw.column_names}")
print("\n--- Sample ---")
print(raw[0])

# Check distribution
from collections import Counter
intent_counts = Counter(json.loads(x["output"])["intent"] for x in samples)
mode_counts = Counter(json.loads(x["output"])["answer_mode"] for x in samples)
print("\nIntent distribution:", intent_counts)
print("Answer mode distribution:", mode_counts)


In [ ]:
# Export dataset ra JSONL để nộp kèm hoặc dùng lại sau này
DATASET_PATH = os.path.join(OUTPUT_DIR, "admissions_router_dataset_v1.jsonl")
with open(DATASET_PATH, "w", encoding="utf-8") as f:
    for row in samples:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"✓ Saved dataset: {DATASET_PATH}")

# Validate JSON output schema
required_keys = {
    "intent", "answer_mode", "needs_retrieval", "needs_clarification",
    "clarification_question", "transformed_query"
}
for i, row in enumerate(samples):
    obj = json.loads(row["output"])
    assert set(obj.keys()) == required_keys, (i, obj.keys())
    assert obj["intent"] in INTENTS, (i, obj["intent"])
    assert obj["answer_mode"] in ANSWER_MODES, (i, obj["answer_mode"])
print("✓ All outputs are valid router JSON")


## 2. Format Alpaca + Token Length Analysis

Model được train theo template:

```text
### Instruction:
...

### Input:
User question

### Response:
JSON route
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

MODEL_NAME = "unsloth/Qwen2.5-3B-bnb-4bit"
MAX_SEQ_CAP = 1024

ALPACA_TEMPLATE = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

ALPACA_TEMPLATE_FOR_INFERENCE = """### Instruction:
{instruction}

### Input:
{input}

### Response:
"""

def format_alpaca(example):
    text = ALPACA_TEMPLATE.format(
        instruction=example["instruction"],
        input=example["input"],
        output=example["output"],
    )
    return {"text": text}

ds = raw.map(format_alpaca, remove_columns=raw.column_names)
print("--- Formatted sample ---")
print(ds[0]["text"][:1000])

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
lengths = [len(tok.encode(x["text"])) for x in ds]
p50 = int(np.percentile(lengths, 50))
p95 = int(np.percentile(lengths, 95))
p99 = int(np.percentile(lengths, 99))

print("\nToken length distribution:")
print(f"min={min(lengths)}, max={max(lengths)}, p50={p50}, p95={p95}, p99={p99}")

MAX_SEQ_LENGTH = min(MAX_SEQ_CAP, 1 << (max(p95, 256) - 1).bit_length())
print(f"✓ Chọn max_seq_length = {MAX_SEQ_LENGTH}")

plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=30)
plt.axvline(p95, linestyle='--', label=f'p95={p95}')
plt.axvline(MAX_SEQ_LENGTH, linestyle='--', label=f'chosen={MAX_SEQ_LENGTH}')
plt.xlabel('Tokens')
plt.ylabel('Count')
plt.title('Admissions Router Dataset — Token Length')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "token_length_distribution.png"), dpi=160)
plt.show()


In [ ]:
# Split raw trước để giữ target JSON phục vụ router-specific evaluation
split_raw = raw.train_test_split(test_size=0.1, seed=42)
train_raw = split_raw["train"]
eval_raw = split_raw["test"]

train_ds = train_raw.map(format_alpaca, remove_columns=train_raw.column_names)
eval_ds = eval_raw.map(format_alpaca, remove_columns=eval_raw.column_names)

print(f"✓ Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print("Eval sample question:", eval_raw[0]["input"])
print("Eval target:", eval_raw[0]["output"])


## 3. Load Base Model + LoRA Config


In [ ]:
from unsloth import FastLanguageModel

def load_base_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    return model, tokenizer

def wrap_with_lora(model, r, alpha):
    return FastLanguageModel.get_peft_model(
        model,
        r=r,
        lora_alpha=alpha,
        target_modules=["q_proj", "v_proj"],  # đúng lab spec
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

base_model, tokenizer = load_base_model()
model = wrap_with_lora(base_model, r=16, alpha=32)
trainable_16, total_params = count_trainable_params(model)
print(f"✓ Trainable params r=16: {trainable_16:,} ({100*trainable_16/total_params:.4f}% of total)")


## 4. Trainer Helpers

T4 mode:

- batch size = 1
- gradient accumulation = 8
- eval giữa training = tắt để tránh OOM
- eval sau khi train bằng `safe_evaluate()`


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, Trainer
import inspect, trl, transformers

print(f"trl: {trl.__version__} | transformers: {transformers.__version__}")

# Patch compatibility tokenizer -> processing_class cho một số version TRL/Transformers
try:
    import unsloth.models._utils as _u_utils
    _underlying_init = getattr(_u_utils, "_original_trainer_init", Trainer.__init__)
    if not getattr(_underlying_init, "_aliased", False):
        def _aliased_trainer_init(self, *args, **kwargs):
            if "tokenizer" in kwargs and "processing_class" not in kwargs:
                kwargs["processing_class"] = kwargs.pop("tokenizer")
            return _underlying_init(self, *args, **kwargs)
        _aliased_trainer_init._aliased = True
        _u_utils._original_trainer_init = _aliased_trainer_init
        if "tokenizer" not in inspect.signature(Trainer.__init__).parameters:
            _orig_t = Trainer.__init__
            def _t_init(self, *args, **kwargs):
                if "tokenizer" in kwargs and "processing_class" not in kwargs:
                    kwargs["processing_class"] = kwargs.pop("tokenizer")
                return _orig_t(self, *args, **kwargs)
            _t_init._aliased = True
            Trainer.__init__ = _t_init
        print("✓ Trainer.__init__ patched")
except Exception as e:
    print("Patch skipped:", e)

try:
    from trl import SFTConfig
    _HAS_SFTCONFIG = True
except ImportError:
    _HAS_SFTCONFIG = False

_TA_PARAMS = inspect.signature(TrainingArguments.__init__).parameters
_EVAL_KEY = "eval_strategy" if "eval_strategy" in _TA_PARAMS else "evaluation_strategy"
_SFT_PARAMS = inspect.signature(SFTTrainer.__init__).parameters
_SUPPORTS_OLD_KWARGS = "dataset_text_field" in _SFT_PARAMS

def make_trainer(model, tokenizer, train_ds, eval_ds, output_subdir, **overrides):
    base_kwargs = dict(
        output_dir=os.path.join(OUTPUT_DIR, output_subdir),
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        eval_accumulation_steps=4,
        prediction_loss_only=True,
        gradient_accumulation_steps=8,
        warmup_ratio=0.10,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_steps=25,
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=42,
        report_to="none",
    )
    base_kwargs[_EVAL_KEY] = "no"
    base_kwargs.update(overrides)

    if _HAS_SFTCONFIG:
        sft_extra = dict(dataset_text_field="text", packing=False, max_seq_length=MAX_SEQ_LENGTH)
        sft_params = inspect.signature(SFTConfig.__init__).parameters
        sft_extra = {k: v for k, v in sft_extra.items() if k in sft_params}
        valid_base = {k: v for k, v in base_kwargs.items() if k in sft_params}
        args = SFTConfig(**valid_base, **sft_extra)
    else:
        args = TrainingArguments(**base_kwargs)

    trainer_kwargs = {
        "model": model,
        "train_dataset": train_ds,
        "eval_dataset": eval_ds,
        "args": args,
    }
    if "processing_class" in _SFT_PARAMS:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    if _SUPPORTS_OLD_KWARGS:
        trainer_kwargs.update(dict(dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, packing=False))

    return SFTTrainer(**trainer_kwargs)

def safe_evaluate(trainer):
    gc.collect(); torch.cuda.empty_cache()

    try:
        from transformers.utils.notebook import NotebookProgressCallback
        trainer.remove_callback(NotebookProgressCallback)
    except Exception:
        pass

    try:
        return trainer.evaluate()["eval_loss"]
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        print(f"⚠ trainer.evaluate() failed ({type(e).__name__}). Falling back to manual eval...")

    gc.collect(); torch.cuda.empty_cache()
    m = trainer.model
    m.eval()
    eval_dl = trainer.get_eval_dataloader()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in eval_dl:
            batch = {k: v.to(m.device) for k, v in batch.items() if hasattr(v, "to")}
            out = m(**batch)
            total += out.loss.item()
            n += 1
            del out
            torch.cuda.empty_cache()
    return total / max(n, 1)


## 5. Train Baseline `r=16`


In [ ]:
# Train baseline r=16
import numpy as np

torch.cuda.reset_peak_memory_stats()
trainer_16 = make_trainer(model, tokenizer, train_ds, eval_ds, "r16")

t0 = time.time()
trainer_16.train()
wall_16 = time.time() - t0
vram_16 = torch.cuda.max_memory_allocated() / 1e9

trainer_16.save_model(os.path.join(OUTPUT_DIR, "r16"))
print(f"✓ Saved r=16 adapter to {os.path.join(OUTPUT_DIR, 'r16')}")
print(f"✓ r=16 done in {wall_16/60:.1f} min, peak VRAM = {vram_16:.1f} GB")

try:
    eval_loss_16 = safe_evaluate(trainer_16)
    ppl_16 = float(np.exp(eval_loss_16))
    print(f"✓ r=16 eval loss = {eval_loss_16:.4f}, perplexity = {ppl_16:.2f}")
except Exception as e:
    print(f"⚠ Eval failed: {e}")
    eval_loss_16 = float("nan")
    ppl_16 = float("nan")


In [ ]:
# Plot + save training loss curve cho report

def plot_losses(log_history, title="Training Loss", filename="loss_curve_r16.png"):
    df = pd.DataFrame(log_history)
    train = df[df["loss"].notna()] if "loss" in df else pd.DataFrame()
    eval_ = df[df["eval_loss"].notna()] if "eval_loss" in df else pd.DataFrame()

    plt.figure(figsize=(8, 4))
    if not train.empty:
        plt.plot(train["step"], train["loss"], label="train")
    if not eval_.empty:
        plt.plot(eval_["step"], eval_["eval_loss"], label="eval", marker="o")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=160)
    plt.show()
    print(f"✓ Saved loss curve: {path}")

plot_losses(trainer_16.state.log_history, title="Loss Curve — Admissions Router r=16")


## 6. Rank Experiment — Train `r=8` và `r=64`


In [ ]:
def train_one_rank(r, alpha):
    gc.collect(); torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    base_m, tok = load_base_model()
    m = wrap_with_lora(base_m, r=r, alpha=alpha)
    trainable, total = count_trainable_params(m)

    tr = make_trainer(m, tok, train_ds, eval_ds, f"r{r}")
    t0 = time.time()
    tr.train()
    wall = time.time() - t0
    vram = torch.cuda.max_memory_allocated() / 1e9

    adapter_dir = os.path.join(OUTPUT_DIR, f"r{r}")
    tr.save_model(adapter_dir)
    print(f"✓ r={r} adapter saved: {adapter_dir}")

    try:
        eval_loss = safe_evaluate(tr)
    except Exception as e:
        print(f"⚠ Eval failed for r={r}: {e}")
        eval_loss = float("nan")

    metrics = {
        "rank": r,
        "alpha": alpha,
        "trainable_params": int(trainable),
        "train_time_min": wall / 60,
        "peak_vram_gb": vram,
        "eval_loss": eval_loss,
        "eval_perplexity": float(np.exp(eval_loss)) if not np.isnan(eval_loss) else float("nan"),
    }
    return metrics, tr, m


In [ ]:
# Cleanup baseline model trước khi train rank mới
del trainer_16, model, base_model
gc.collect(); torch.cuda.empty_cache()

print("========== Training r=8 ==========")
exp_8, trainer_8, model_8 = train_one_rank(r=8, alpha=16)

del trainer_8, model_8
gc.collect(); torch.cuda.empty_cache()


In [ ]:
print("========== Training r=64 ==========")
exp_64, trainer_64, model_64 = train_one_rank(r=64, alpha=128)

del trainer_64, model_64
gc.collect(); torch.cuda.empty_cache()


In [ ]:
# Build summary table
results = [
    {
        "rank": 16,
        "alpha": 32,
        "trainable_params": int(trainable_16),
        "train_time_min": wall_16 / 60,
        "peak_vram_gb": vram_16,
        "eval_loss": eval_loss_16,
        "eval_perplexity": ppl_16,
    },
    exp_8,
    exp_64,
]
summary_df = pd.DataFrame(results).sort_values("rank").reset_index(drop=True)
summary_path = os.path.join(OUTPUT_DIR, "rank_experiment_summary.csv")
summary_df.to_csv(summary_path, index=False)
print("=== Rank Experiment Summary ===")
print(summary_df.to_string(index=False))
print(f"✓ Saved: {summary_path}")


## 7. Qualitative Comparison — Base vs Fine-tuned Router

Test prompt là các câu hỏi tuyển sinh thật/sát thực tế. Fine-tuned model nên trả JSON route chuẩn hơn base model.


In [ ]:
TEST_QUERIES = [
    "Ngành Khoa học Máy tính học phí bao nhiêu?",
    "Học phí bao nhiêu?",
    "Em muốn biết học bổng VinUni cần điều kiện gì?",
    "Deadline nộp hồ sơ năm 2026 là khi nào?",
    "Ngành Data Science học những gì?",
    "Cảm ơn bạn nhé",
    "Có cần IELTS để vào ngành AI không?",
    "VinUni có ký túc xá không?",
    "hoc phi nganh ai bao nhieu",
    "Cho mình hỏi quy trình apply vào VinUni",
]
print(f"✓ {len(TEST_QUERIES)} test queries")


In [ ]:
from peft import PeftModel


def build_inference_prompt(query):
    return ALPACA_TEMPLATE_FOR_INFERENCE.format(
        instruction=ROUTER_INSTRUCTION,
        input=query,
    )


def generate_router_output(model, tokenizer, query, max_new_tokens=256):
    FastLanguageModel.for_inference(model)
    prompt = build_inference_prompt(query)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return full.split("### Response:")[-1].strip()


def extract_json_object(text):
    text = text.strip()
    # Nếu model có nói thừa, cố tìm object JSON đầu tiên
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

# Reload base + r16 adapter
base_for_eval, tok_for_eval = load_base_model()
ft_model = PeftModel.from_pretrained(base_for_eval, os.path.join(OUTPUT_DIR, "r16"))

qualitative_results = []
for i, query in enumerate(TEST_QUERIES[:10], 1):
    print(f"\n━━━ Query {i}: {query}")
    base_resp = generate_router_output(base_for_eval, tok_for_eval, query)
    ft_resp = generate_router_output(ft_model, tok_for_eval, query)
    qualitative_results.append({
        "query": query,
        "base_output": base_resp,
        "finetuned_r16_output": ft_resp,
        "finetuned_json_valid": extract_json_object(ft_resp) is not None,
    })
    print("BASE:", base_resp[:300])
    print("FT  :", ft_resp[:300])

qual_df = pd.DataFrame(qualitative_results)
qual_path = os.path.join(OUTPUT_DIR, "qualitative_comparison.csv")
qual_df.to_csv(qual_path, index=False)
print(f"\n✓ Saved qualitative comparison: {qual_path}")


## 8. Router-specific Evaluation

Phần này không bắt buộc trong rubric, nhưng rất hợp với product chatbot tuyển sinh:

- JSON validity
- intent accuracy
- answer_mode accuracy
- needs_retrieval accuracy
- needs_clarification accuracy


In [ ]:
def compare_router_fields(pred, target):
    fields = ["intent", "answer_mode", "needs_retrieval", "needs_clarification"]
    return {f"{f}_correct": pred.get(f) == target.get(f) for f in fields}

router_eval_rows = []
for row in eval_raw:
    query = row["input"]
    target = json.loads(row["output"])
    pred_text = generate_router_output(ft_model, tok_for_eval, query)
    pred = extract_json_object(pred_text)
    valid = pred is not None

    result = {
        "query": query,
        "target": row["output"],
        "prediction_text": pred_text,
        "json_valid": valid,
    }
    if valid:
        result.update(compare_router_fields(pred, target))
    else:
        result.update({
            "intent_correct": False,
            "answer_mode_correct": False,
            "needs_retrieval_correct": False,
            "needs_clarification_correct": False,
        })
    router_eval_rows.append(result)

router_eval_df = pd.DataFrame(router_eval_rows)
metrics = {
    "json_valid_rate": router_eval_df["json_valid"].mean(),
    "intent_accuracy": router_eval_df["intent_correct"].mean(),
    "answer_mode_accuracy": router_eval_df["answer_mode_correct"].mean(),
    "needs_retrieval_accuracy": router_eval_df["needs_retrieval_correct"].mean(),
    "needs_clarification_accuracy": router_eval_df["needs_clarification_correct"].mean(),
}

metrics_path = os.path.join(OUTPUT_DIR, "router_eval_metrics.json")
router_eval_path = os.path.join(OUTPUT_DIR, "router_eval_details.csv")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
router_eval_df.to_csv(router_eval_path, index=False)

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print(f"✓ Saved: {metrics_path}")
print(f"✓ Saved: {router_eval_path}")


## 9. Optional — Push Adapter lên HuggingFace Hub


In [ ]:
# Optional: push r16 adapter lên HuggingFace Hub để nộp Option B hoặc deploy model server
PUSH_TO_HUB = False
HUB_REPO_ID = "your-username/admissions-router-qwen2p5-3b-r16"

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()
    ft_model.push_to_hub(HUB_REPO_ID)
    tok_for_eval.push_to_hub(HUB_REPO_ID)
    print(f"✓ Adapter pushed: https://huggingface.co/{HUB_REPO_ID}")
else:
    print("Skip push_to_hub. Đổi PUSH_TO_HUB=True nếu muốn upload adapter.")


## 10. Checklist nộp bài

Trong `OUTPUT_DIR` cần có:

- `admissions_router_dataset_v1.jsonl`
- `r8/` adapter checkpoint
- `r16/` adapter checkpoint
- `r64/` adapter checkpoint
- `rank_experiment_summary.csv`
- `qualitative_comparison.csv`
- `router_eval_metrics.json`
- `router_eval_details.csv`
- `loss_curve_r16.png`
- `token_length_distribution.png`

Phần report sẽ viết dựa trên các file này.

### Gợi ý kết luận report

Với product chatbot tuyển sinh, rank tốt nhất thường là `r=8` hoặc `r=16` vì router là task output ngắn, schema cố định. Nếu `r=64` chỉ giảm perplexity rất ít nhưng tốn VRAM/thời gian hơn nhiều, nên không đáng để deploy. Dùng `r=16` làm baseline an toàn; nếu muốn tối ưu chi phí thì thử `r=8`.
